In [1]:
%reload_ext autoreload
%autoreload 2
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme(theme="default")

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

In [2]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """生成数据画像，每个字段输出一行统计信息。

    缺失值包括 NaN/None/NA，以及文本类字段（str、object、category）中
    去除首尾空白后为空的字符串。唯一值个数和示例值均基于非缺失值计算，
    保证各统计项的缺失口径一致。

    Args:
        df: 待分析的 DataFrame。允许 0 行、重复列名，以及包含 list、dict
            等不可哈希值的列。

    Returns:
        每个字段一行的 DataFrame，包含以下列：

        - 字段：列名。
        - 类型：pandas dtype 的字符串表示，如 "int64"、"str"。
        - 缺失数：缺失值个数（int）。
        - 缺失率：缺失数 / 总行数（float），0 行时为 0.0。
        - 唯一值个数：非缺失值的去重个数（int）。不可哈希的值按其字符串
        形式去重。
        - 示例值：第一个非缺失值；全部缺失时为 None。
    """
    n = len(df)
    rows = []
    for col, s in df.items():  # 按位置取列，列名重复也不会出错
        missing = s.isna()
        if (pd.api.types.is_string_dtype(s) or s.dtype == object
                or isinstance(s.dtype, pd.CategoricalDtype)):
            blank = s.astype("string").str.strip().eq("").fillna(False).astype(bool)
            missing = missing | blank
        valid = s[~missing]
        try:
            n_unique = int(valid.nunique())
        except TypeError:  # 列里有 list/dict 等不可哈希的值
            n_unique = int(valid.astype(str).nunique())
        n_missing = int(missing.sum())
        rows.append({
            "字段": col,
            "类型": str(s.dtype),
            "缺失数": n_missing,
            "缺失率": n_missing / n if n else 0.0,
            "唯一值个数": n_unique,
            "示例值": valid.iloc[0] if len(valid) else None,
        })
    return pd.DataFrame(rows)

In [ ]:
cell = pd.read_sql("SELECT * FROM ods_cell", con)
from signal_insight.quality import profile
profile(cell)

In [ ]:
from matplotlib.path import Path as Mppath
def rings_from_points(points, key):
    pts = points if "ring" in points.columns else points.assign(ring=0)
    return {
        name:[
            r.sort_values("seq")[['lng', 'lat']].to_numpy(dtype=float)
            for _, r in g.groupby("ring")
        ]
        for name, g in pts.groupby(key, sort=False)
    }

def point_in_ploygons(lng, lat, polys):
    xy = np.column_stack(
        [np.asarray(tng, dtype=float),
         np.asarray(lat, dtype=float)])
    out = np.full(len(xy), None, dtype=object)
    for name, rings in polys.items():
        inside = np.zeros(len(xy), dtype=bool)
        for ring in rings:
            inside = Mppath(ring).contains_points(xy)
    return out

districts = rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")

sites['districts'] = point_in_ploygons(sites['Ing'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")